In [ ]:
!wget -q https://epss.empiricalsecurity.com/epss_scores-current.csv.gz
!wget -q https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.csv
!for y in 2019 2020 2021 2022 2023 2024 2025 2026; do wget -q https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-$y.json.gz; done
!ls -lh

total 160M
-rw-r--r-- 1 root root 2.6M Oct 11 13:30 epss_scores-current.csv.gz
-rw-r--r-- 1 root root 9.8M Oct  9 07:15 nvdcve-2.0-2019.json.gz
-rw-r--r-- 1 root root  14M Oct  9 07:15 nvdcve-2.0-2020.json.gz
-rw-r--r-- 1 root root  17M Oct 10 07:15 nvdcve-2.0-2021.json.gz
-rw-r--r-- 1 root root  18M Oct 10 07:15 nvdcve-2.0-2022.json.gz
-rw-r--r-- 1 root root  20M Oct 10 07:15 nvdcve-2.0-2023.json.gz
-rw-r--r-- 1 root root  24M Oct 10 07:15 nvdcve-2.0-2024.json.gz
-rw-r--r-- 1 root root  24M Oct 11 07:15 nvdcve-2.0-2025.json.gz
-rw-r--r-- 1 root root  34M Oct 11 07:15 nvdcve-2.0-2026.json.gz
drwxr-xr-x 1 root root 4.0K Oct  2 13:50 sample_data


In [ ]:
!wget -q https://raw.githubusercontent.com/cisagov/kev-data/main/known_exploited_vulnerabilities.csv
!wc -l known_exploited_vulnerabilities.csv

1740 known_exploited_vulnerabilities.csv


In [ ]:
import gzip, json, glob
import pandas as pd

# --- NVD: one row per CVE with CVSS v3.1 data ---
rows = []
for f in sorted(glob.glob("nvdcve-2.0-*.json.gz")):
    with gzip.open(f) as fh:
        data = json.load(fh)
    for v in data["vulnerabilities"]:
        c = v["cve"]
        m = c.get("metrics", {}).get("cvssMetricV31")
        if not m:
            continue  # skip CVEs without CVSS 3.1
        m = next((x for x in m if x.get("type") == "Primary"), m[0])
        d = m["cvssData"]
        cwes = [x["value"] for w in c.get("weaknesses", []) for x in w["description"]]
        refs = c.get("references", [])
        rows.append({
            "cve": c["id"],
            "published": c["published"],
            "base_score": d["baseScore"],
            "exploitability": m.get("exploitabilityScore"),
            "impact": m.get("impactScore"),
            "attack_vector": d["attackVector"],
            "attack_complexity": d["attackComplexity"],
            "privileges": d["privilegesRequired"],
            "user_interaction": d["userInteraction"],
            "scope": d["scope"],
            "conf": d["confidentialityImpact"],
            "integ": d["integrityImpact"],
            "avail": d["availabilityImpact"],
            "cwe": cwes[0] if cwes else "NONE",
            "n_refs": len(refs),
            "has_exploit_ref": any("Exploit" in r.get("tags", []) for r in refs),
            "has_patch_ref": any("Patch" in r.get("tags", []) for r in refs),
        })
nvd = pd.DataFrame(rows)

# --- EPSS + KEV ---
epss = pd.read_csv("epss_scores-current.csv.gz", comment="#")
kev = set(pd.read_csv("known_exploited_vulnerabilities.csv")["cveID"])

df = nvd.merge(epss[["cve", "epss"]], on="cve", how="left")
df["epss"] = df["epss"].fillna(0)
df["in_kev"] = df["cve"].isin(kev)

# --- Label: actually exploited (KEV) or high EPSS ---
df["label"] = (df["in_kev"] | (df["epss"] >= 0.1)).astype(int)

df.to_parquet("cves.parquet")
print(df.shape)
print(df["label"].value_counts())
print(df.head(3).T)


(249200, 20)
label
0    242826
1      6374
Name: count, dtype: int64
                                         0                        1  \
cve                          CVE-2019-3500            CVE-2019-5009   
published          2019-01-02T07:29:00.197  2019-01-04T14:29:00.237   
base_score                             7.8                      7.2   
exploitability                         1.8                      1.2   
impact                                 5.9                      5.9   
attack_vector                        LOCAL                  NETWORK   
attack_complexity                      LOW                      LOW   
privileges                             LOW                     HIGH   
user_interaction                      NONE                     NONE   
scope                            UNCHANGED                UNCHANGED   
conf                                  HIGH                     HIGH   
integ                                 HIGH                     HIGH   
avail   

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_parquet("cves.parquet")

# --- Features ---
df["age_days"] = (pd.Timestamp.now() - pd.to_datetime(df["published"])).dt.days
top_cwes = df["cwe"].value_counts().index[:30]
df["cwe"] = df["cwe"].where(df["cwe"].isin(top_cwes), "OTHER")

cat_cols = ["attack_vector","attack_complexity","privileges","user_interaction",
            "scope","conf","integ","avail","cwe"]
num_cols = ["base_score","exploitability","impact","n_refs","age_days",
            "has_exploit_ref","has_patch_ref"]

X = pd.get_dummies(df[cat_cols + num_cols], columns=cat_cols).astype(float)
y = df["label"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# --- Baseline: just rank by CVSS score ---
base = X_te["base_score"]
print(f"CVSS baseline   ROC-AUC {roc_auc_score(y_te, base):.3f}  PR-AUC {average_precision_score(y_te, base):.3f}")

# --- Model 1: Logistic regression ---
mu, sd = X_tr.mean(), X_tr.std().replace(0, 1)
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit((X_tr - mu) / sd, y_tr)
p_lr = lr.predict_proba((X_te - mu) / sd)[:, 1]
print(f"LogReg          ROC-AUC {roc_auc_score(y_te, p_lr):.3f}  PR-AUC {average_precision_score(y_te, p_lr):.3f}")

# --- Model 2: Gradient boosting ---
gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=42)
gb.fit(X_tr, y_tr)
p_gb = gb.predict_proba(X_te)[:, 1]
print(f"GradBoost       ROC-AUC {roc_auc_score(y_te, p_gb):.3f}  PR-AUC {average_precision_score(y_te, p_gb):.3f}")

# --- Practical metric: of the top 5% riskiest CVEs flagged, how many are actually exploited? ---
k = int(len(y_te) * 0.05)
for name, s in [("CVSS", base.values), ("LogReg", p_lr), ("GradBoost", p_gb)]:
    top = np.argsort(-s)[:k]
    print(f"{name:10s} precision@top5%: {y_te.values[top].mean():.3f}  "
          f"recall@top5%: {y_te.values[top].sum() / y_te.sum():.3f}")

CVSS baseline   ROC-AUC 0.723  PR-AUC 0.060
LogReg          ROC-AUC 0.839  PR-AUC 0.164
GradBoost       ROC-AUC 0.880  PR-AUC 0.232
CVSS       precision@top5%: 0.075  recall@top5%: 0.147
LogReg     precision@top5%: 0.181  recall@top5%: 0.355
GradBoost  precision@top5%: 0.219  recall@top5%: 0.428


In [ ]:
cut = pd.Timestamp("2025-01-01")
pub = pd.to_datetime(df["published"])
tr, te = pub < cut, pub >= cut

gb_t = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=42)
gb_t.fit(X[tr], y[tr])
p = gb_t.predict_proba(X[te])[:, 1]

k = int(te.sum() * 0.05)
for name, s in [("CVSS", X[te]["base_score"].values), ("GradBoost", p)]:
    top = np.argsort(-s)[:k]
    print(f"{name:10s} PR-AUC {average_precision_score(y[te], s):.3f}  "
          f"recall@top5% {y[te].values[top].sum() / y[te].sum():.3f}")
print("test positives:", y[te].sum(), "of", te.sum())

CVSS       PR-AUC 0.028  recall@top5% 0.194
GradBoost  PR-AUC 0.037  recall@top5% 0.269
test positives: 1015 of 112925


In [ ]:
X2 = X.drop(columns=["age_days"])
tr, te = pub < "2024-01-01", (pub >= "2024-01-01") & (pub < "2025-01-01")

gb_t = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=42)
gb_t.fit(X2[tr], y[tr])
p = gb_t.predict_proba(X2[te])[:, 1]

k = int(te.sum() * 0.05)
print("random PR-AUC ~", round(y[te].mean(), 3))
for name, s in [("CVSS", X2[te]["base_score"].values), ("GradBoost", p)]:
    top = np.argsort(-s)[:k]
    print(f"{name:10s} PR-AUC {average_precision_score(y[te], s):.3f}  "
          f"recall@top5% {y[te].values[top].sum() / y[te].sum():.3f}")

random PR-AUC ~ 0.024
CVSS       PR-AUC 0.058  recall@top5% 0.167
GradBoost  PR-AUC 0.098  recall@top5% 0.219


In [ ]:
import joblib, sklearn

final = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=42)
final.fit(X2, y)

joblib.dump({
    "model": final,
    "columns": list(X2.columns),
    "top_cwes": list(top_cwes),
}, "cve_model.joblib")

print("sklearn version:", sklearn.__version__)
!ls -lh cve_model.joblib

sklearn version: 1.6.1
-rw-r--r-- 1 root root 476K Oct 12 00:58 cve_model.joblib


In [ ]:
feat_cols = cat_cols + [c for c in num_cols if c != "age_days"]
df[["cve"] + feat_cols].to_csv("nvd_features.csv.gz", index=False)
!ls -lh nvd_features.csv.gz

-rw-r--r-- 1 root root 1.8M Oct 12 01:00 nvd_features.csv.gz
